In [13]:
import cv2
import operator as op
import sys
import copy

img_test = cv2.imread("Images de test pour la decetion de visages/image_test.jpg")
img_test = cv2.resize(img_test,(0,0),fx=0.25,fy=0.25)
img_test = cv2.cvtColor(img_test,cv2.COLOR_BGR2GRAY)
ff = cv2.imread("Images de test pour la decetion de visages/ff.jpg")
ff = cv2.cvtColor(ff,cv2.COLOR_BGR2GRAY)
tt = cv2.imread("Images de test pour la decetion de visages/tt.png")
tt = cv2.cvtColor(tt,cv2.COLOR_BGR2GRAY)

img_list = [img_test,ff,tt]

tt_flipped=cv2.flip(tt,1) #on retourne horizontalement les images avec des profiles
ff_flipped=cv2.flip(ff,1)
img_list.append(tt_flipped)
img_list.append(ff_flipped)

def overlap(face,profile): #fonction de detection des chavauchements
    x1,y1,w1,h1 = face
    x2,y2,w2,h2 = profile

    return not (x1 + w1 < x2 or x2 + w2 < x1 or y1 + h1 < y2 or y2 + h2 < y1)

cv2.imshow("img_test",img_test) 
cv2.imshow("ff",ff)
cv2.imshow("tt",tt)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [25]:
face_cascade = cv2.CascadeClassifier("classifieurs/haarcascade_frontalface_alt2.xml")
profile_cascade = cv2.CascadeClassifier("classifieurs/haarcascade_profileface.xml")

all_faces = []

img_list_copy = copy.deepcopy(img_list) #pas nécessaire mais plus facile pour déboguer (pas besoin de recharger les images et on a pas les rectangles des essais précédents)

idx = 0
for img in img_list_copy:
    img_mir = cv2.flip(img,1)
    #faces = face_cascade.detectMultiScale(img,1.05,18) #Paramètres ok ne plus toucher
    faces = face_cascade.detectMultiScale(img,1.15,6)
    #profiles = profile_cascade.detectMultiScale(img,1.01,60) #Paramètres médiocres mais j'ai pas trouvé mieux
    profiles = profile_cascade.detectMultiScale(img,1.1,3) 
    
    # Détection des profils pour l'image miroir
    profiles_mir = profile_cascade.detectMultiScale(img_mir,1.1,3)
    w_img = len(img[0])
    h_img = len(img)
    # Si un profil est détecté dans l'image miroir on place les coordinnées du rectangle au 
    # bon endroit sur l'image de base.
    for p in profiles_mir:
        p[0] = w_img - p[0] - p[2]
    
    profiles_all = list(profiles) + list(profiles_mir)
    
    no_ol_profiles =[] #detection des chevauchements
    for p in profiles_all:
        if not any(overlap(f,p) for f in faces):
            no_ol_profiles.append(p)

    list_faces = list(faces) + no_ol_profiles
    all_faces.append(list_faces)

    img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)


    for(x,y,w,h) in list_faces: #dessin des rectangles
        x2,y2 = x+w , y+h
        cv2.rectangle(img_color, (x, y), (x2, y2), (0, 0, 255), thickness=6)

    cv2.imshow(f"Image_{idx}", img_color)

    idx +=1

total_faces = sum(len(faces) for faces in all_faces) 
print(f"Nombre total de visages détectés : {total_faces}")

cv2.waitKey(0)
cv2.destroyAllWindows()

Nombre total de visages détectés : 20
